In [30]:
!pip install -q langchain langchain-groq

import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # LangChain reads it from here

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.3)

In [31]:

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

SYSTEM = """ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80 each.
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time or place. If it isn't in FACTS,
        say "I'll check with the owner." Work out totals carefully.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),          # the chat so far goes here
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

history = []
def ask(question: str) -> str:
    answer = chain.invoke({"history": history, "question": question})
    history.extend([("human", question), ("ai", answer)])
    return answer

print(ask("Hi, I'm Ravi. Do you make eggless cakes?"))
print(ask("What's my name?"))

Hi Ravi! Yes, all our cakes—plum, chocolate, and cupcakes—have eggless versions. Let me know what you’d like, and I’ll check availability for you.
Your name is Ravi.


In [32]:
import csv

def run_tests(path: str = "tests.csv") -> None:
    rows = list(csv.DictReader(open(path)))
    passed = 0
    for row in rows:
        answer = chain.invoke({"history": [], "question": row["question"]})
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"], "→", answer[:70])
    print(f"\n{passed}/{len(rows)} passed ({passed / len(rows):.0%})")

run_tests()

PASS | How much is a plum cake? → A plum cake costs Rs 500.
PASS | Do you make eggless cakes? → Yes, we offer eggless versions of all our cakes – plum, chocolate, and
FAIL | Can I get a cake today evening? → We need 24 hours’ notice for orders. I’ll check with the owner.
PASS | Do you deliver to Kakinada? → I’ll check with the owner.
PASS | What is the price of 6 cupcakes? → The price of 6 cupcakes is Rs 480.

4/5 passed (80%)


In [10]:
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[], system_prompt=SYSTEM)   # tools come tomorrow
result = agent.invoke({"messages": [{"role": "user", "content": "Do you make eggless cakes?"}]})
print(result["messages"][-1].content)

Yes, all our cakes have eggless versions.
